# Install Requirements and Validate Dataset

## Install Library

In [ ]:
!pip install -r requirements.txt

## Dataset check

In [ ]:
DATASET_LOCATION = "FlyingChairs_release"

In [2]:
!echo "Number of files in dataset"
!ls "{DATASET_LOCATION}/data" | wc -l

Number of files in dataset
68616


# Pre-requisites

## Imports

In [ ]:
import os, io, random, struct, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import imageio

## Define values

In [ ]:
# Dataset
DATA_ROOT   = DATASET_LOCATION
SUBSET_SIZE = 22872                                     # number of images you want to train on --- in this case i have set it to entire dataset size
                                                        # can be automated but later ... maybe 
VAL_SPLIT   = 0.1                                       # validation split ratio --- should probably incrase to 0.3 (70-30)
RESOLUTION  = 256                                       # resolution of the images essentially controls the spatial detail vs mem cost 

# Training
BATCH_SIZE    = 32                                      # training batch size --- greater the value more vram needed
EPOCHS        = 60                                      # do i really have to explain this? (number of epochs to train the model)
LOG_INTERVAL  = 100                                     # steps between logging values in output ... does not affect training at all

# Architecture
BASE_CH      = 96                                       # Base number of feature channels in encoder
MAX_DISP     = 4                                        # cost volume search radius: (2*4+1)**2 = 81 channels (standard RAFT value)
N_GRU_ITERS  = 6                                        # number of itterations for the convGRU --- higher value makes the inference slower but better refinement

# Optimisation
LR            = 1e-4                                    # learning rate cold start value
WEIGHT_DECAY  = 1e-6                                    # L2 regularization on weights
SCALE_WEIGHTS = [1.00, 0.50, 0.25]                      # Loss weights at different scales. Higher weight on coarse helps stabilize early training.
SMOOTHNESS_W  = 0.0001                                  # Regularizes flow field to be smooth, basically a control for how sharp the edges should be (should probably decrease it)
GRAD_W        = 0.02                                    # gradient loss weight

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

## Dataset validation

In [ ]:
# .FLO reader
def read_flo_file(path: str) -> np.ndarray:
    """Load .flo binary file into float32 (H, W, 2) array."""
    MAGIC = 202021.25
    fsize = os.path.getsize(path)
    if fsize < 12:
        raise ValueError(f"Corrupt .flo (too small: {fsize} bytes): {path}")
    with open(path, 'rb') as f:
        magic = struct.unpack('f', f.read(4))[0]
        if abs(magic - MAGIC) > 1e-3:
            raise ValueError(f"Bad .flo magic: {path}")
        w, h  = struct.unpack('ii', f.read(8))
        expected = h * w * 2 * 4
        data = f.read()
        if len(data) < expected:
            raise ValueError(f"Truncated .flo ({len(data)}/{expected} bytes): {path}")
        flow  = np.frombuffer(data, dtype=np.float32).reshape(h, w, 2)
    return flow



# Triplet discovery
def discover_triplets(root: str, max_samples: int = 8000, verbose: bool = True):
    data_dir = os.path.join(root, 'data') if os.path.isdir(os.path.join(root, 'data')) else root
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Data dir not found: {data_dir!r}")

    all_files  = sorted(os.listdir(data_dir))
    base_names = sorted(
        {f.replace('_img1.ppm', '') for f in all_files if f.endswith('_img1.ppm')}
    )

    triplets = []
    for base in base_names:
        p1 = os.path.join(data_dir, base + '_img1.ppm')
        p2 = os.path.join(data_dir, base + '_img2.ppm')
        pf = os.path.join(data_dir, base + '_flow.flo')
        if os.path.exists(p1) and os.path.exists(p2) and os.path.exists(pf):
            triplets.append((base, p1, p2, pf))
        if len(triplets) >= max_samples:
            break

    if verbose:
        print(f"Valid triplets found : {len(triplets):,}  (cap={max_samples:,})")
    return triplets


triplets = discover_triplets(DATA_ROOT, max_samples=SUBSET_SIZE)

n_val          = int(len(triplets) * VAL_SPLIT)
n_train        = len(triplets) - n_val
train_triplets = triplets[:n_train]
val_triplets   = triplets[n_train:]
print(f"Train: {n_train:,} | Val: {n_val:,}")

Valid triplets found : 22,872  (cap=22,872)
Train: 20,585 | Val: 2,287


# Augmentation Class

In [ ]:
class OpticalFlowAugmentor:
    def __init__(self, output_size=256, do_hflip=True, do_vflip=False):
        self.output_size = output_size
        self.do_hflip = do_hflip
        self.do_vflip = do_vflip

    def __call__(self, img1, img2, flow):
        # img1, img2: numpy HWC (uint8)
        # flow: numpy HWC (float32)

        S = self.output_size
        h, w = img1.shape[:2]

        # random crop
        scale = np.random.uniform(0.7, 1.0)
        ch, cw = int(h * scale), int(w * scale)
        y0 = np.random.randint(0, h - ch + 1)
        x0 = np.random.randint(0, w - cw + 1)

        img1 = img1[y0:y0+ch, x0:x0+cw]
        img2 = img2[y0:y0+ch, x0:x0+cw]
        flow = flow[y0:y0+ch, x0:x0+cw]

        # resize
        img1 = cv2.resize(img1, (S, S))
        img2 = cv2.resize(img2, (S, S))
        flow = cv2.resize(flow, (S, S))

        flow[..., 0] *= (S / cw)
        flow[..., 1] *= (S / ch)

        # flips
        if self.do_hflip and np.random.rand() < 0.5:
            img1 = img1[:, ::-1].copy()    # flip W idth axis
            img2 = img2[:, ::-1].copy()
            flow = flow[:, ::-1].copy()
            flow[..., 0] *= -1              # negate horizontal flow

        if self.do_vflip and np.random.rand() < 0.5:
            img1 = img1[::-1, :].copy()    # flip H eight axis
            img2 = img2[::-1, :].copy()
            flow = flow[::-1, :].copy()
            flow[..., 1] *= -1              # negate vertical flow

        # photometric augmentations
        # brightness / contrast
        if np.random.rand() < 0.8:
            alpha = np.random.uniform(0.7, 1.3)
            beta = np.random.uniform(-20, 20)
            img1 = np.clip(img1 * alpha + beta, 0, 255)
            img2 = np.clip(img2 * alpha + beta, 0, 255)

        # gaussian blur
        if np.random.rand() < 0.3:
            k = np.random.choice([3, 5])
            img1 = cv2.GaussianBlur(img1, (k, k), 0)
            img2 = cv2.GaussianBlur(img2, (k, k), 0)

        # noise
        if np.random.rand() < 0.5:
            noise = np.random.randn(*img1.shape) * 5
            img1 = np.clip(img1 + noise, 0, 255)
            img2 = np.clip(img2 + noise, 0, 255)

        # to tensor
        img1 = torch.from_numpy(img1).permute(2, 0, 1).float() / 255.0
        img2 = torch.from_numpy(img2).permute(2, 0, 1).float() / 255.0
        flow = torch.from_numpy(flow).permute(2, 0, 1).float()

        return img1, img2, flow

In [ ]:
## Image Normalisation
#Resizing the tensorts using bilinear interpolation and rescale displacement vectors accordingly
def resize_flow_tensor(flow: torch.Tensor, h: int, w: int) -> torch.Tensor:
    _, _, H, W = flow.shape
    flow_r = F.interpolate(flow, size=(h, w), mode='bilinear', align_corners=False)
    flow_r[:, 0] *= w / W
    flow_r[:, 1] *= h / H
    return flow_r


# ImageNet normalisation
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def normalize_imgs(img: torch.Tensor) -> torch.Tensor:
    m = _MEAN.to(img.device)
    s = _STD.to(img.device)
    return (img - m) / s


## Dataset loaders

In [ ]:
class FlyingChairsDataset(Dataset):
    def __init__(self, triplets, augmentor=None, resolution=256):
        self.triplets   = triplets
        self.augmentor  = augmentor
        self.resolution = resolution

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        _, p1, p2, pf = self.triplets[idx]
    
        try:
            img1 = cv2.imread(p1, cv2.IMREAD_COLOR)
            img2 = cv2.imread(p2, cv2.IMREAD_COLOR)
    
            if img1 is None or img2 is None:
                raise ValueError(f"Failed to read image: {p1} or {p2}")
    
            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
    
            flow = read_flo_file(pf)  # H x W x 2
    
        except (ValueError, OSError) as e:
            # Skip corrupt sample → move to next index
            return self.__getitem__((idx + 1) % len(self))
    
        # Augmentation path
        if self.augmentor is not None:
            return self.augmentor(img1, img2, flow)
    
        # Resize + scale flow
        S = self.resolution
        oh, ow = flow.shape[:2]
    
        img1 = cv2.resize(img1, (S, S), interpolation=cv2.INTER_LINEAR)
        img2 = cv2.resize(img2, (S, S), interpolation=cv2.INTER_LINEAR)
        flow = cv2.resize(flow, (S, S), interpolation=cv2.INTER_LINEAR)
    
        flow[..., 0] *= (S / ow)
        flow[..., 1] *= (S / oh)
    
        # To tensor
        img1 = torch.from_numpy(img1).permute(2, 0, 1).float() / 255.0
        img2 = torch.from_numpy(img2).permute(2, 0, 1).float() / 255.0
        flow = torch.from_numpy(flow).permute(2, 0, 1).float()
    
        return img1, img2, flow

In [ ]:
augmentor    = OpticalFlowAugmentor(output_size=RESOLUTION)                                             # Augmentor object
train_ds     = FlyingChairsDataset(train_triplets, augmentor=augmentor, resolution=RESOLUTION)          # Training dataset object
val_ds       = FlyingChairsDataset(val_triplets,   augmentor=None,      resolution=RESOLUTION)          # Validation dataset object

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=12,                                                                                    # Adjust according to your GPU ... this was set for A100 so you get the refference
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Actual Model definations

### Residual Block

In [ ]:
"""
a standard residual (skip-connection) block for refining features without losing original information
allows block to learn residual mapping (difference from input) instead of full transformation
helps preserve low-level spatial details

Data flow:
    Conv - > BatchNorm -> Relu -> Conv - > BatchNorm
                                                     }->  Relu ................. (Both convolution layers are 3x3)
                                    Skip Connections 
"""
class ResidualBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.net(x) + x)


### Encoder Block

In [ ]:
"""
Encoder extracts hierarchical features and downsamples the image to a coarse representation (H/4) while preserving intermediate features for skip connections.
Data flow:
    x → block1 → s1
      → block2 → s2
      → block3 → c
      → block4 → block5 → f

Returns:
    f  : coarse features (H/4, 2*base_ch)
    s2 : skip features (H/4, base_ch)
    s1 : skip features (H/2, base_ch)

"""
class Encoder(nn.Module):
   
    def __init__(self, in_ch: int = 3, base_ch: int = 64):
        
        super().__init__()
        
        def _layer(inc, outc, stride, k):                                               # Everylayer is : Convolutional layer, with BatchNorm and Relu
            return nn.Sequential(
                nn.Conv2d(inc, outc, k, stride=stride, padding=k//2, bias=False),
                nn.BatchNorm2d(outc), nn.ReLU(inplace=True))

        self.block1 = _layer(in_ch, base_ch, stride=2, k=7)                             # convolution layer .. Kernel size 7x7, stride 2 , downsamples to H/2
        self.block2 = _layer(base_ch, base_ch, stride=2, k=5)                           # convolution layer .. Kernel size 5x5, stride 2 , downsamples to H/4
        self.block3 = _layer(base_ch, base_ch*2, stride=1, k=3)                         # no downsampling .. increase channels from 64 to 128 and produce coarse feature map

        self.block4 = nn.Sequential(                                                    # Refine the features at H/4
            _layer(base_ch*2, base_ch*2, stride=1, k=3),
            ResidualBlock(base_ch*2),                                                   # Preseve spatial information while improving representaion
        )

        self.block5 = nn.Sequential(
            ResidualBlock(base_ch * 2),
            ResidualBlock(base_ch * 2),
        )


    def forward(self, x):
        s1 = self.block1(x)                                                             # [B, 64,  H/2, W/2]
        s2 = self.block2(s1)                                                            # [B, 64,  H/4, W/4]
        c  = self.block3(s2)                                                            # [B, 128, H/4, W/4]
        f = self.block5(self.block4(c))                                                 # [B, 128, H/4, W/4]
        return f, s2, s1                    # coarse, skip from H/4, skip from H/2


### Cost Volume

In [ ]:
"""
Builds a correlation volume between two feature maps (f1, f2) by comparing each pixel in f1 with a neighborhood in f2
Vectorised L2-normalised correlation cost volume .... makes the dot product behave like cosine similarity
Data flow:
    f1, f2 → L2 normalize
           → pad f2
           → for each (dy, dx) in search window:
                 shift f2 → compute dot product with f1
           → stack all correlations → cost volume

Output:
    corr : [B, (2*max_disp+1)^2, H, W]

Output: [B, (2·max_disp+1)^2, H, W]
"""
class CostVolume(nn.Module):
    def __init__(self, max_disp: int = 4, stride: int = 1, dilation: int = 1):
        super().__init__()
        
        self.max_disp = max_disp
        self.stride   = stride
        self.dilation = dilation
        
        # Pre-compute displacement offsets
        disps = list(range(-max_disp, max_disp + 1, stride))
        self._offsets = [(dy, dx) for dy in disps for dx in disps]

    def forward(self, f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
        f1 = F.normalize(f1, p=2, dim=1)                                        # normalise f1
        f2 = F.normalize(f2, p=2, dim=1)                                        # normalise f2
        B, C, H, W = f1.shape                                                   # batch size, channels, heigth, width
        pad = self.max_disp * self.dilation
        f2p = F.pad(f2, [pad] * 4)
        corrs = []                                                              # corelation maps
        for dy, dx in self._offsets:
            f2_shifted = f2p[:, :, pad + dy*self.dilation : pad + dy*self.dilation + H,
                                   pad + dx*self.dilation : pad + dx*self.dilation + W]
            corrs.append((f1 * f2_shifted).sum(dim=1))

        corr = torch.stack(corrs, dim=1)                                        # stacking into the final cost volume
        return corr

"""
Compress wide cost-volume channel dimension to compact feature before GRU processing
Data flow:
    cost volume → Conv → BN → ReLU
                → Conv → BN → ReLU

Output:
    compressed feature map with reduced channels
"""
class CostVolumeEncoder(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 64):
        super().__init__()
        mid = max(in_ch // 2, out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, mid,    3, padding=1, bias=False), nn.BatchNorm2d(mid),    nn.ReLU(inplace=True),
            nn.Conv2d(mid,   out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


### convGRU

In [ ]:
"""
a recurrent unit that refines spatial features iteratively while preserving spatial structure.

"""
class ConvGRUCell(nn.Module):
    def __init__(self, input_ch: int, hidden_ch: int, kernel: int = 3):
        super().__init__()
        pad = kernel // 2
        combined = input_ch + hidden_ch
        self.update = nn.Conv2d(combined, hidden_ch, kernel, padding=pad)
        self.reset  = nn.Conv2d(combined, hidden_ch, kernel, padding=pad)
        self.cand   = nn.Conv2d(combined, hidden_ch, kernel, padding=pad)
        self.hidden_ch = hidden_ch

    def forward(self, x: torch.Tensor, h: torch.Tensor = None) -> torch.Tensor:
        if h is None:
            h = x.new_zeros(x.size(0), self.hidden_ch, x.size(2), x.size(3))
        xh = torch.cat([x, h], dim=1)
        z  = torch.sigmoid(self.update(xh))                                         # update gate
        r  = torch.sigmoid(self.reset(xh))                                          # reset gate
        q  = torch.tanh(self.cand(torch.cat([x, r * h], dim=1)))                    # candidate state
        return (1 - z) * h + z * q

### Decoder

In [ ]:
"""
Decoder with skip connections from img1's encoder.
Produces flow at three scales for multi-scale supervision.

Returns (flow_s2, flow_s1, flow_full) at (H/4, H/2, H) respectively.
"""
class FlowDecoder(nn.Module):
    def __init__(self, gru_ch: int = 64, skip2_ch: int = 64, skip1_ch: int = 64):
        super().__init__()
        def _block(inc, outc):
            return nn.Sequential(
                nn.Conv2d(inc, outc, 3, padding=1, bias=False),
                nn.BatchNorm2d(outc), nn.ReLU(inplace=True))

        # H/4 -> H/4 refine
        self.up1     = _block(gru_ch, 64)
        self.fuse1   = _block(64 + skip2_ch,64)
        self.head_s2 = nn.Conv2d(64, 2, 3, padding=1)

        # H/4 -> H/2
        self.up2     = _block(64, 32)
        self.fuse2   = _block(32 + skip1_ch, 32)
        self.head_s1 = nn.Conv2d(32, 2, 3, padding=1)

        # H/2 -> H
        self.up3       = _block(32, 16)
        self.head_full = nn.Conv2d(16, 2, 3, padding=1)

    def forward(self, h, skip2, skip1):
        x = self.up1(h)                                                                         # GRU hidden → 64
        x = self.fuse1(torch.cat([x, skip2], dim=1))                                            # cat skip2
        flow_s2 = self.head_s2(x)                                                               # flow at h/4

        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)              # increment to H/2 using bilinear inter 
        x = self.fuse2(torch.cat([self.up2(x), skip1], dim=1))                                  # cat skip1
        flow_s1 = self.head_s1(x)                                                               # flow at h/2

        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        flow_full = self.head_full(self.up3(x))

        return flow_s2, flow_s1, flow_full


### Optic Flow Pipeline

In [ ]:
# if you have followed comments till now, this should be pretty self explainatory

class OpticalFlowNet(nn.Module):
    def __init__(self, base_ch: int = 64, max_disp: int = 8, n_iters: int = 6):
        
        super().__init__()
        n_cv_ch       = (2 * max_disp + 1) ** 2
        self.encoder  = Encoder(in_ch=3, base_ch=base_ch)
        self.cost_vol = CostVolume(max_disp=max_disp)
        self.cv_enc   = CostVolumeEncoder(in_ch=n_cv_ch, out_ch=base_ch)
        
        # flow_enc compresses current flow estimate into feature
        self.flow_enc = nn.Sequential(
            nn.Conv2d(2, base_ch // 2, 7, padding=3, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch // 2, base_ch // 2, 3, padding=1, bias=False),
            nn.ReLU(inplace=True),
        )
        
        # GRU input: f1(2*base_ch) + cost_volume(base_ch) + flow(base_ch//2)
        self.gru     = ConvGRUCell(input_ch=base_ch * 2 + base_ch // 2 + base_ch,
                                   hidden_ch=base_ch)
        self.decoder = FlowDecoder(gru_ch=base_ch, skip2_ch=base_ch, skip1_ch=base_ch)
        self.n_iters = n_iters
        
        # Coarse flow initialiser (1x1 conv from GRU hidden)
        self.flow_head_coarse = nn.Conv2d(base_ch, 2, 1)

    def forward(self, img1, img2):
        f1, s2, s1 = self.encoder(img1)
        f2, _,  _  = self.encoder(img2)

        B, C, Hc, Wc = f1.shape

        # Initialise flow at coarse scale (H/4)
        flow_coarse = torch.zeros(B, 2, Hc, Wc, device=img1.device)
        h = None

        for _ in range(self.n_iters):
            
            # Warp f2 by current flow estimate at coarse scale --- defined below
            f2_warped = warp_features(f2, flow_coarse)

            # Recompute cost volume from updated f2
            cv    = self.cost_vol(f1, f2_warped)
            cv_f  = self.cv_enc(cv)

            # Encode current flow
            flow_f = self.flow_enc(flow_coarse)

            # Fuse all context for GRU
            fused  = torch.cat([f1, cv_f, flow_f], dim=1)
            h      = self.gru(fused, h)

            # Update coarse flow residual
            flow_coarse = flow_coarse + self.flow_head_coarse(h)

        return self.decoder(h, s2, s1)


### Attention Gate

In [ ]:
"""
Instead of blindly using skip connections, this gate learns where to focus
Data flow:
    skip, context → upsample(context)
                  → concatenate(skip, context)
                  → 1x1 Conv → Sigmoid → attention map g
                  → element-wise multiply: skip * g

Output:
    gated skip features (filtered spatially)
"""
class AttentionGate(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(ch * 2, ch, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, skip, context):
        context_up = F.interpolate(context, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        g = self.gate(torch.cat([skip, context_up], dim=1))
        return skip * g

# Global Code

## Backward warping

In [ ]:
"""
Backward-warp `feat` using `flow`.
feat : [B, C, H, W]
flow : [B, 2, H, W] flow in PIXELS at the feat resolution
Data flow:
    feat + flow → build coordinate grid
                → add flow (pixel displacement)
                → normalize to [-1, 1]
                → grid_sample (bilinear interpolation)

Output:
    warped feature map aligned using flow
"""
def warp_features(feat: torch.Tensor, flow: torch.Tensor) -> torch.Tensor:
    B, C, H, W = feat.shape

    # Build normalised sampling grid [-1, 1]
    grid_y, grid_x = torch.meshgrid(
        torch.arange(H, dtype=feat.dtype, device=feat.device),
        torch.arange(W, dtype=feat.dtype, device=feat.device),
        indexing="ij",
    )
    grid = torch.stack([grid_x, grid_y], dim=0).unsqueeze(0)                # [1, 2, H, W]
    grid = grid.expand(B, -1, -1, -1)

    # Add flow and normalise to [-1, 1]
    sample_x = (grid[:, 0] + flow[:, 0]) / (W - 1) * 2 - 1                  # [B, H, W]
    sample_y = (grid[:, 1] + flow[:, 1]) / (H - 1) * 2 - 1

    # expects [B, H, W, 2]
    sample_grid = torch.stack([sample_x, sample_y], dim=-1)

    return F.grid_sample(
        feat, sample_grid,
        mode="bilinear",
        padding_mode="border",                                              # border avoids black edges at frame boundaries
        align_corners=True,
    )


def warp_image(img: torch.Tensor, flow: torch.Tensor) -> torch.Tensor:
    return warp_features(img, flow)

## SSIM helper

In [ ]:
"""
Computes a per-pixel Structural Similarity Map(SSIM) between two images
Computes:
       luminance (mean)
       contrast (variance)
       structure (covariance)

"""
def _ssim_map(x: torch.Tensor, y: torch.Tensor, window: int = 3) -> torch.Tensor:
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    pad = window // 2
    
    # luminance (mean)
    mu_x  = F.avg_pool2d(x, window, stride=1, padding=pad)
    mu_y  = F.avg_pool2d(y, window, stride=1, padding=pad)
    mu_xy = mu_x * mu_y
    mu_x2 = mu_x ** 2
    mu_y2 = mu_y ** 2

    # Variances (contrast)
    sig_x2  = F.avg_pool2d(x * x, window, stride=1, padding=pad) - mu_x2
    sig_y2  = F.avg_pool2d(y * y, window, stride=1, padding=pad) - mu_y2

    # Covariance (structure)
    sig_xy  = F.avg_pool2d(x * y, window, stride=1, padding=pad) - mu_xy

    # ((2 * μ_x * μ_y + C1) * (2 * σ_xy + C2)) / ((μ_x² + μ_y² + C1) * (σx² + σy² + C2))
    ssim = ((2 * mu_xy + C1) * (2 * sig_xy + C2)) / \
           ((mu_x2 + mu_y2 + C1) * (sig_x2 + sig_y2 + C2))
    
    return ssim.clamp(0, 1)

## Photometric loss

In [ ]:
"""
Photometric reconstruction loss for optical flow.

Given flow from img1 → img2:
    - Warp img2 back into img1’s frame
    - Compare reconstructed image with img1

Loss = alpha * SSIM + (1 - alpha) * Charbonnier L1

Notes:
    - Only one image is warped (img2). img1 is the reference.
    - Mask removes invalid (out-of-bounds) warped pixels.
    - SSIM captures structural similarity; L1 captures pixel accuracy.
"""
def photometric_loss(
    img1:         torch.Tensor,   # [B, 3, H, W] unnormalised [0,1]
    img2:         torch.Tensor,   # [B, 3, H, W]
    flow_pred:    torch.Tensor,   # [B, 2, H, W]
    alpha:        float = 0.85,   # SSIM vs L1 blend
) -> torch.Tensor:

    img2_warped = warp_image(img2, flow_pred)                                           # Warp img2 into img1 frame using predicted flow

    valid = ((img2_warped > 0) & (img2_warped < 1)).all(dim=1, keepdim=True).float()    # keeps only pixels that map inside image bounds

    l1 = ((img2_warped - img2) ** 2 + 1e-4).sqrt()                                      # Charbonnier L1

                                                                                        
    ssim_map = _ssim_map(img2_warped, img2)                                             # SSIM loss (1 - SSIM)
    ssim_loss = (1.0 - ssim_map) / 2.0                                                  # normalize to [0, 1]

    # Combine losses: alpha * SSIM + (1-alpha) * L1
    photo = alpha * ssim_loss.mean(dim=1, keepdim=True) + (1.0 - alpha) * l1.mean(dim=1, keepdim=True)

    loss = (valid * photo).sum() / (valid.sum() + 1e-6)                                # Masked average loss
    return loss


## Updated multiscale loss with photometric term

In [ ]:
"""
Multi-scale loss with EPE + smoothness + gradient + photometric (SSIM+L1).

- EPE: supervised flow error
- Smoothness: encourages spatial consistency (edge-aware)
- Gradient: preserves motion boundaries
- Photometric: enforces image reconstruction via warping (img2 → img1)

Photometric loss is applied only at finer scales (H/2, H) where flow is reliable.
"""

PHOTO_W = 0.25                                      # photometric loss weight

def multiscale_loss_with_photo(
    pred_flows: tuple,                              # (flow_s2, flow_s1, flow_full)
    flow_gt:    torch.Tensor,
    img1:       torch.Tensor,
    img2:       torch.Tensor,                       
    weights:    list  = (1.00, 0.50, 0.25),
    smooth_w:   float = 0.0001,
    grad_w:     float = 0.02,
    photo_w:    float = 0.10,
) -> tuple:

    total = flow_gt.new_zeros(1)
    epe_full = 0.0

    for i, (pred, w) in enumerate(zip(pred_flows, weights)):
        _, _, h, ww = pred.shape

        # Resize GT flow and img1 to current prediction scale
        gt_r = resize_flow_tensor(flow_gt, h, ww)
        img_r = F.interpolate(img1, (h, ww), mode='bilinear', align_corners=False)

        # EPE
        l_epe = epe(pred, gt_r)

        # Edge-aware smoothness
        l_smt = edge_aware_smoothness(pred, img_r)        # defined later

        # Gradient consistency: aligns spatial flow changes with GT (preserves edges)
        dx_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
        dy_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]
        dx_gt = gt_r[:, :, :, 1:] - gt_r[:, :, :, :-1]
        dy_gt = gt_r[:, :, 1:, :] - gt_r[:, :, :-1, :]
        l_grad = (dx_pred - dx_gt).abs().mean() + (dy_pred - dy_gt).abs().mean()

        # sum supervised losses
        total = total + w * (l_epe + smooth_w * l_smt + grad_w * l_grad)

        # Photometric loss at finest 2 scales ... Skip coarsest scale (flow too rough for meaningful warping)
        if i >= 1:                                                                      # flow_s1 (H/2) and flow_full (H)
            img2_r = F.interpolate(img2, (h, ww), mode='bilinear', align_corners=False)
            l_photo = photometric_loss(img_r, img2_r, pred)
            total = total + w * photo_w * l_photo

        # Log full-res EPE
        if i == len(pred_flows) - 1:
            epe_full = l_epe.item()

    return total.squeeze(), epe_full

In [ ]:
model = OpticalFlowNet(base_ch=BASE_CH, max_disp=MAX_DISP, n_iters=N_GRU_ITERS).to(device)
if hasattr(torch, 'compile'):                           # first compile run will be slower, but after that it should be much faster
    model = torch.compile(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params/1e6:.2f}M")

# Forward-pass sanity check
with torch.no_grad():
    _x = torch.randn(2, 3, RESOLUTION, RESOLUTION, device=device)
    _y = torch.randn(2, 3, RESOLUTION, RESOLUTION, device=device)
    fs2, fs1, ff = model(_x, _y)
    print(f"flow_s2  shape : {tuple(fs2.shape)}")                                       # expected (2, 2, 64,  64)
    print(f"flow_s1  shape : {tuple(fs1.shape)}")                                       # expected (2, 2, 128, 128)
    print(f"flow_full shape: {tuple(ff.shape)}")                                        # expected (2, 2, 256, 256)
    assert ff.shape == (2, 2, RESOLUTION, RESOLUTION), "Output resolution mismatch"
print("Sanity check passed")

## Visualisation methods -- Ai used

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn.functional as F
from matplotlib.colors import hsv_to_rgb


# ── Correct HSV flow colour (unchanged, already correct) ─────────────────────
def flow_to_color(flow_hw2: np.ndarray) -> np.ndarray:
    u, v   = flow_hw2[:, :, 0], flow_hw2[:, :, 1]
    mag    = np.sqrt(u ** 2 + v ** 2)
    angle  = np.arctan2(v, u)
    hue    = (angle / (2 * np.pi) + 0.5) % 1.0
    sat    = np.clip(mag / (mag.max() + 1e-6), 0, 1)
    val    = np.ones_like(hue)
    return (hsv_to_rgb(np.stack([hue, sat, val], -1).astype(np.float32)) * 255).astype(np.uint8)


# ── Shared-scale magnitude visualisation ─────────────────────────────────────
def visualize_flow_comparison(
    img1_t:      torch.Tensor,   # [3, H, W] uint8 or [0,1]
    pred_flow_t: torch.Tensor,   # [2, H, W]
    gt_flow_t:   torch.Tensor,   # [2, H, W]
    tag: str = "epoch000",
    out_dir: str = "./diagnostics",
    cmap: str = "viridis",
):
    """
    Saves a 2×3 figure:
      Row 0: Input | Predicted flow (HSV) | Predicted magnitude
      Row 1: Error map (EPE) | GT flow (HSV) | GT magnitude
    GT and Pred magnitude share identical vmin/vmax so they're directly comparable.
    Error map uses its own scale anchored at 0.
    """
    img_np  = img1_t.permute(1, 2, 0).cpu().float().clamp(0, 1).numpy()
    pred_np = pred_flow_t.permute(1, 2, 0).detach().cpu().numpy()   # H,W,2  float
    gt_np   = gt_flow_t.permute(1, 2, 0).cpu().numpy()

    # ── Magnitudes (float, NOT scaled to 255) ────────────────────────────────
    pred_mag = np.sqrt(pred_np[..., 0] ** 2 + pred_np[..., 1] ** 2)
    gt_mag   = np.sqrt(gt_np[..., 0]   ** 2 + gt_np[..., 1]   ** 2)

    # Shared scale: use GT max so clipping is semantically meaningful
    shared_vmax = float(gt_mag.max()) + 1e-6
    shared_vmin = 0.0

    # ── EPE per pixel ─────────────────────────────────────────────────────────
    diff   = pred_np - gt_np          # H,W,2
    epe_map = np.sqrt((diff ** 2).sum(-1))   # H,W  in pixels

    # ── HSV colour wheels ─────────────────────────────────────────────────────
    pred_rgb = flow_to_color(pred_np)
    gt_rgb   = flow_to_color(gt_np)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(f"Flow diagnostics — {tag}", fontsize=12, fontweight="bold")

    def _show(ax, data, title, **kw):
        im = ax.imshow(data, **kw)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        return im

    # Row 0
    _show(axes[0, 0], img_np,   "Input frame 1")
    _show(axes[0, 1], pred_rgb, "Predicted flow (HSV)")
    im_pm = _show(axes[0, 2], pred_mag, "Predicted magnitude (px)",
                  cmap=cmap, vmin=shared_vmin, vmax=shared_vmax)
    fig.colorbar(im_pm, ax=axes[0, 2], fraction=0.046, pad=0.04,
                 label="displacement (px)")

    # Row 1
    _show(axes[1, 0], epe_map,  "EPE per pixel (↑ = worse)",
          cmap="hot", vmin=0, vmax=epe_map.max())
    fig.colorbar(axes[1, 0].images[0], ax=axes[1, 0],
                 fraction=0.046, pad=0.04, label="EPE (px)")
    _show(axes[1, 1], gt_rgb,   "GT flow (HSV)")
    im_gm = _show(axes[1, 2], gt_mag, "GT magnitude (px)",
                  cmap=cmap, vmin=shared_vmin, vmax=shared_vmax)
    fig.colorbar(im_gm, ax=axes[1, 2], fraction=0.046, pad=0.04,
                 label="displacement (px)")

    mean_epe = epe_map.mean()
    fig.text(0.5, 0.01, f"Mean EPE = {mean_epe:.3f} px | "
             f"GT max = {gt_mag.max():.2f} px | Pred max = {pred_mag.max():.2f} px",
             ha="center", fontsize=9, color="gray")

    plt.tight_layout()
    path = f"{out_dir}/diagnostic_{tag}.png"
    plt.savefig(path, dpi=100, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}  (mean EPE = {mean_epe:.3f} px)")
    return mean_epe


# ── Histogram comparison ──────────────────────────────────────────────────────
def plot_magnitude_histograms(pred_flow_t, gt_flow_t, tag="epoch000", out_dir="./histogram"):
    pred_np = pred_flow_t.permute(1, 2, 0).detach().cpu().numpy()
    gt_np   = gt_flow_t.permute(1, 2, 0).cpu().numpy()
    pred_mag = np.sqrt(pred_np[..., 0] ** 2 + pred_np[..., 1] ** 2).ravel()
    gt_mag   = np.sqrt(gt_np[..., 0]   ** 2 + gt_np[..., 1]   ** 2).ravel()

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(0, max(gt_mag.max(), pred_mag.max()) + 1, 60)
    ax.hist(gt_mag,   bins=bins, alpha=0.6, label="GT", color="#1f77b4")
    ax.hist(pred_mag, bins=bins, alpha=0.6, label="Predicted", color="#ff7f0e")
    ax.set_xlabel("Flow magnitude (pixels)"); ax.set_ylabel("Pixel count")
    ax.set_title(f"Magnitude distribution — {tag}")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/hist_{tag}.png", dpi=100, bbox_inches="tight")
    plt.close()

## Multi-scale supervised loss

In [ ]:
"""
Mean end-point error with small epsilon for numerical stability
"""
def epe(pred: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.sum((pred - gt) ** 2, dim=1) + 1e-6).mean()

"""
penalises flow gradients, weighted by image gradients so edges are preserved
"""
def edge_aware_smoothness(flow: torch.Tensor, img: torch.Tensor, alpha: float = 150.0) -> torch.Tensor:
    
    # Flow spatial gradients
    fd_x = (flow[:, :, :, 1:] - flow[:, :, :, :-1]).abs()
    fd_y = (flow[:, :, 1:, :] - flow[:, :, :-1, :]).abs()
    # Image edge weights (grayscale)
    gray = img.mean(dim=1, keepdim=True)
    id_x = (gray[:, :, :, 1:] - gray[:, :, :, :-1]).abs()
    id_y = (gray[:, :, 1:, :] - gray[:, :, :-1, :]).abs()
    w_x  = torch.exp(-alpha * id_x)
    w_y  = torch.exp(-alpha * id_y)
    return (w_x * fd_x).mean() + (w_y * fd_y).mean()


def multiscale_loss(
    pred_flows : tuple,
    flow_gt    : torch.Tensor,
    img1       : torch.Tensor,
    weights    : list  = (0.10, 0.30, 1.00),
    smooth_w   : float = 0.01,
    grad_w     : float = 0.1
) -> tuple:

    total    = flow_gt.new_zeros(1)
    epe_full = 0.0
    for i, (pred, w) in enumerate(zip(pred_flows, weights)):
        _, _, h, ww = pred.shape

        # Resize GT + image
        gt_r  = resize_flow_tensor(flow_gt, h, ww)
        img_r = F.interpolate(img1, (h, ww), mode='bilinear', align_corners=False)

        # Base losses
        l_epe = epe(pred, gt_r)
        l_smt = edge_aware_smoothness(pred, img_r)

        # Gradient loss
        dx_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
        dy_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]

        dx_gt = gt_r[:, :, :, 1:] - gt_r[:, :, :, :-1]
        dy_gt = gt_r[:, :, 1:, :] - gt_r[:, :, :-1, :]

        l_grad = (dx_pred - dx_gt).abs().mean() + (dy_pred - dy_gt).abs().mean()

        # Total
        total = total + w * (l_epe + smooth_w * l_smt + grad_w * l_grad)

        # Log full-res EPE
        if i == len(pred_flows) - 1:
            epe_full = l_epe.item()

    return total, epe_full


## Training

In [ ]:
final_optic_flow_model_name = "heh"

In [ ]:
"""
I get that this is not very production like, but
Please note that you will have to run this code block twice, once with the scratch values, 
Second time with the fine tuning values
the values are mentioned beside the relevant variables in comments
"""

from tqdm import tqdm
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


"""
Load the previously trained model if exists
"""
import os
_ckpt_path = f'{final_optic_flow_model_name}.pt'
if os.path.exists(_ckpt_path):
    _ckpt = torch.load(_ckpt_path, map_location=device, weights_only=False)
    try:
        model.load_state_dict(_ckpt['model_state'])
        best_val_epe = _ckpt.get('val_epe', float('inf'))
        print(f'Loaded checkpoint: epoch={_ckpt["epoch"]}, val_epe={best_val_epe:.4f}')
        print('Starting Phase 2: fine-tuning with warping loss')
    except Exception as e:
        best_val_epe = float('inf')
        print('Architecture mismatch (probably H/4 update) training from scratch')
else:
    best_val_epe = float('inf')
    print('No checkpoint found training from scratch with warping loss')


# Fine-tuning scheduler: lower max_lr since model if 
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,                                            # for training from scratch : 3e-4,  fine-tuning phase = 1.5e-4
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,                                          # 0.1 to 0.2 for scratch training, 0.05 for fine tuning
    div_factor=10,
    final_div_factor=1000,
    anneal_strategy='cos'
)
train_history = {'loss': [], 'epe': []}
val_history   = {'loss': [], 'epe': []}

for epoch in range(1, EPOCHS + 1):

    # Train
    model.train()
    t_losses, t_epes = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

    for step, (img1, img2, flow_gt) in enumerate(pbar, 1):
      img1, img2, flow_gt = img1.to(device), img2.to(device), flow_gt.to(device)

      optimizer.zero_grad()

      # Mixed precision forward
      with autocast("cuda"):
          pred_flows = model(normalize_imgs(img1), normalize_imgs(img2))
          loss, ep   = multiscale_loss_with_photo(
              pred_flows, flow_gt, img1, img2,
              weights=SCALE_WEIGHTS,
              smooth_w=SMOOTHNESS_W,
              grad_w=GRAD_W,
              photo_w=PHOTO_W,
          )

      #  Scaled backward
      scaler.scale(loss).backward()

      # Gradient clipping (needs unscale first)
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

      # Optimizer step
      scaler.step(optimizer)
      scaler.update()

      scheduler.step()

      t_losses.append(loss.item())
      t_epes.append(ep)

      pbar.set_postfix({
          "loss": f"{np.mean(t_losses[-20:]):.3f}",
          "epe":  f"{np.mean(t_epes[-20:]):.3f}",
          "lr":   f"{scheduler.get_last_lr()[0]:.1e}"
      })

    # Validation
    model.eval()
    v_losses, v_epes = [], []

    vbar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for img1, img2, flow_gt in vbar:
            img1, img2, flow_gt = img1.to(device), img2.to(device), flow_gt.to(device)

            with autocast("cuda"):
                pred_flows = model(normalize_imgs(img1), normalize_imgs(img2))
                # Use same loss as training for fair comparison
                loss, ep = multiscale_loss_with_photo(
                    pred_flows, flow_gt, img1, img2,
                    weights=SCALE_WEIGHTS,
                    smooth_w=SMOOTHNESS_W,
                    grad_w=GRAD_W,
                    photo_w=PHOTO_W,
                )

            v_losses.append(loss.item())
            v_epes.append(ep.item() if torch.is_tensor(ep) else ep)

            vbar.set_postfix({
                "val_loss": f"{np.mean(v_losses):.3f}",
                "val_epe":  f"{np.mean(v_epes):.3f}"
            })
    mtr_l, mtr_e = np.mean(t_losses), np.mean(t_epes)
    mvl_l, mvl_e = np.mean(v_losses), np.mean(v_epes)
    train_history['loss'].append(mtr_l); train_history['epe'].append(mtr_e)
    val_history['loss'].append(mvl_l);   val_history['epe'].append(mvl_e)

    print(f"Epoch {epoch:3d} | Train loss={mtr_l:.4f} epe={mtr_e:.4f} | ",
          f"Val loss={mvl_l:.4f} epe={mvl_e:.4f}")

    
    # Best model checkpoint
    if mvl_e < best_val_epe:
        best_val_epe = mvl_e
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimizer.state_dict(),
            'val_epe'     : best_val_epe,
            'config'      : {
                'BASE_CH'    : BASE_CH,
                'MAX_DISP'   : MAX_DISP,
                'N_GRU_ITERS': N_GRU_ITERS,
                'RESOLUTION' : RESOLUTION,
            },
        }, 'flow_model.pt')
        print(f"New best val EPE: {best_val_epe:.4f}  saved flow_model.pt")
        with torch.no_grad():
            _i1, _i2, _gt = next(iter(val_loader))
            _i1 = _i1.to(device, non_blocking=True)
            _i2 = _i2.to(device, non_blocking=True)
            _gt = _gt.to(device, non_blocking=True)
            _pred_flows = model(normalize_imgs(_i1), normalize_imgs(_i2))
            _ff = _pred_flows[-1]
            visualize_flow_comparison(
                _i1[0], _ff[0], _gt[0],
                tag=f"epoch{epoch:03d}"
            )
            plot_magnitude_histograms(_ff[0], _gt[0], tag=f"epoch{epoch:03d}")
print(f"\nDone. Best val EPE = {best_val_epe:.4f} px")

In [ ]:
torch.save({
    'epoch'       : EPOCHS,
    'model_state' : model.state_dict(),
    'train_history': train_history,
    'val_history'  : val_history,
    'config'       : {
        'BASE_CH'    : BASE_CH,
        'MAX_DISP'   : MAX_DISP,
        'N_GRU_ITERS': N_GRU_ITERS,
        'RESOLUTION' : RESOLUTION,
        'BATCH_SIZE' : BATCH_SIZE,
        'EPOCHS'     : EPOCHS,
    },
}, f'{final_optic_flow_model_name}.pt')
print(f"Saved {final_optic_flow_model_name}.pt")